In [1]:
import wikipediaapi
import requests
from bs4 import BeautifulSoup
import json
import os
import re

def get_citations_by_scraping(url):
    """Sử dụng BeautifulSoup để lấy nội dung trong mục Chú thích/Tham khảo"""
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        citations = []
        ref_list = soup.find('ol', {'class': 'references'}) or soup.find('div', {'class': 'reflist'})
        if ref_list:
            for li in ref_list.find_all('li'):
                text = li.get_text(separator=" ").strip()
                clean_text = text.lstrip('^ ').strip()
                citations.append(clean_text)
        return citations
    except:
        return []

def get_all_sections_recursive(sections):
    sect_data = []
    for s in sections:
        sect_data.append({
            "title": s.title,
            "text": s.text.strip(),
            "subsections": get_all_sections_recursive(s.sections)
        })
    return sect_data

def parse_names(line):
    """
    Xử lý dòng văn bản để trích xuất các tên tiềm năng.
    Ví dụ: 'Trần Cảnh , tức Trần Thái Tông (hoàng đế đầu tiên)' 
    -> ['Trần Cảnh', 'Trần Thái Tông']
    """
    # 1. Loại bỏ nội dung trong ngoặc đơn (thường là chú thích chức danh)
    line = re.sub(r'\(.*?\)', '', line)
    
    # 2. Thay thế các từ khóa nối bằng dấu phẩy để dễ split
    line = line.replace(' , ', ',').replace(' tức ', ',').replace(' - ', ',')
    
    # 3. Tách bằng dấu phẩy hoặc khoảng trắng dư thừa (2 khoảng trắng trở lên)
    parts = re.split(r',|\s{2,}', line)
    
    # 4. Làm sạch và loại bỏ các chuỗi rỗng hoặc quá ngắn
    clean_names = []
    for p in parts:
        name = p.strip()
        if len(name) > 2: # Tránh các ký tự rác
            clean_names.append(name)
            
    # Loại bỏ trùng lặp nhưng giữ thứ tự
    return list(dict.fromkeys(clean_names))

def process_historical_figures(input_file, output_file):
    wiki = wikipediaapi.Wikipedia(
        user_agent='HistoricalNLPProject/1.0',
        language='vi',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    results = []

    if not os.path.exists(input_file):
        print(f"Không tìm thấy file: {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        potential_names = parse_names(line)
        found_page = None
        tried_names = []

        print(f"\n--- Đang xử lý dòng: {line}")
        
        # Thử từng tên cho đến khi tìm thấy trang Wikipedia
        for name in potential_names:
            tried_names.append(name)
            page = wiki.page(name)
            if page.exists():
                found_page = page
                print(f"  [OK] Tìm thấy dữ liệu với tên: '{name}'")
                break
            else:
                print(f"  [?] Thử tên '{name}' không thành công...")

        if found_page:
            content_structure = get_all_sections_recursive(found_page.sections)
            citations = get_citations_by_scraping(found_page.fullurl)

            data = {
                "input_line": line,
                "identified_names": potential_names,
                "matched_name": found_page.title,
                "summary": found_page.summary.strip(),
                "full_content_text": found_page.text.strip(),
                "content_hierarchy": content_structure,
                "citations": citations,
                "url": found_page.fullurl
            }
            results.append(data)
        else:
            print(f"  [!] Thất bại: Không tìm thấy bất kỳ tên nào trong {tried_names} trên Wikipedia.")

    with open(output_file, 'w', encoding='utf-8') as jf:
        json.dump(results, jf, ensure_ascii=False, indent=4)

    print(f"\n==========================================")
    print(f"Đã hoàn thành! Kết quả lưu tại: {output_file}")

if __name__ == "__main__":
    process_historical_figures("danh_sach_vua.txt", "data_lich_su_chi_tiet.json")


--- Đang xử lý dòng: Kinh Dương Vương
  [OK] Tìm thấy dữ liệu với tên: 'Kinh Dương Vương'

--- Đang xử lý dòng: Lạc Long Quân  (Hùng Hiền Vương)
  [OK] Tìm thấy dữ liệu với tên: 'Lạc Long Quân'

--- Đang xử lý dòng: Hùng Lân vương
  [?] Thử tên 'Hùng Lân vương' không thành công...
  [!] Thất bại: Không tìm thấy bất kỳ tên nào trong ['Hùng Lân vương'] trên Wikipedia.

--- Đang xử lý dòng: Hùng Diệp vương
  [?] Thử tên 'Hùng Diệp vương' không thành công...
  [!] Thất bại: Không tìm thấy bất kỳ tên nào trong ['Hùng Diệp vương'] trên Wikipedia.

--- Đang xử lý dòng: Hùng Hi vương
  [?] Thử tên 'Hùng Hi vương' không thành công...
  [!] Thất bại: Không tìm thấy bất kỳ tên nào trong ['Hùng Hi vương'] trên Wikipedia.

--- Đang xử lý dòng: Hùng Huy vương
  [?] Thử tên 'Hùng Huy vương' không thành công...
  [!] Thất bại: Không tìm thấy bất kỳ tên nào trong ['Hùng Huy vương'] trên Wikipedia.

--- Đang xử lý dòng: Hùng Chiêu vương
  [?] Thử tên 'Hùng Chiêu vương' không thành công...
  [!] Thất bạ

In [ ]:
import wikipediaapi
import requests
from bs4 import BeautifulSoup
import json
import os
import re

def clean_wiki_text(text):
    """Loại bỏ các ký hiệu chú thích như [1], [2] và khoảng trắng thừa"""
    return re.sub(r'\[\d+\]', '', text).strip()

def get_infobox_data(url):
    """Cào dữ liệu từ bảng Infobox (Hộp thông tin) của trang Wikipedia"""
    info = {}
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find('table', {'class': 'infobox'})
        
        if table:
            rows = table.find_all('tr')
            for row in rows:
                header = row.find('th')
                data = row.find('td')
                if header and data:
                    key = header.get_text(" ", strip=True).replace('\xa0', ' ')
                    value = clean_wiki_text(data.get_text(" ", strip=True))
                    info[key] = value
        return info
    except:
        return {}

def parse_names(line):
    line = re.sub(r'\(.*?\)', '', line)
    line = line.replace(' , ', ',').replace(' tức ', ',').replace(' - ', ',')
    parts = re.split(r',|\s{2,}', line)
    return list(dict.fromkeys([p.strip() for p in parts if len(p.strip()) > 2]))

def process_historical_figures(input_file, output_file):
    wiki = wikipediaapi.Wikipedia(
        user_agent='HistoricalNLPProject/1.1',
        language='vi',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    results = []
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        potential_names = parse_names(line)
        found_page = None
        
        for name in potential_names:
            page = wiki.page(name)
            if page.exists():
                found_page = page
                break

        if found_page:
            print(f"Đang trích xuất: {found_page.title}")
            
            # Lấy Infobox chi tiết
            infobox = get_infobox_data(found_page.fullurl)
            
            # Ánh xạ các trường quan trọng (Tùy biến theo nhu cầu)
            personal_info = {
                "sinh": infobox.get("Sinh", "Không rõ"),
                "mat": infobox.get("Mất", "Không rõ"),
                "trieu_dai": infobox.get("Triều đại", infobox.get("Kỷ nguyên", "Không rõ")),
                "than_phu": infobox.get("Thân phụ", "Không rõ"),
                "than_mau": infobox.get("Thân mẫu", "Không rõ"),
                "tien_nhiem": infobox.get("Tiền nhiệm", "Không rõ"),
                "ke_nhiem": infobox.get("Kế nhiệm", "Không rõ"),
                "an_tang": infobox.get("An táng", "Không rõ")
            }

            results.append({
                "name_in_list": line,
                "wiki_title": found_page.title,
                "personal_info": personal_info, # Thông tin cá nhân đã tách trường
                "raw_infobox": infobox,         # Giữ toàn bộ infobox để phòng hờ
                "summary": found_page.summary.strip(),
                "url": found_page.fullurl
            })

    with open(output_file, 'w', encoding='utf-8') as jf:
        json.dump(results, jf, ensure_ascii=False, indent=4)

if __name__ == "__main__":
    process_historical_figures("danh_sach_vua.txt", "data_lich_su_chi_tiet.json")

Đang trích xuất: Kinh Dương vương
Đang trích xuất: Lạc Long Quân
Đang trích xuất: An Dương Vương
Đang trích xuất: Triệu Vũ Vương
Đang trích xuất: Triệu Văn Vương
Đang trích xuất: Triệu Minh Vương
Đang trích xuất: Triệu Ai Vương
Đang trích xuất: Triệu Dương Vương
Đang trích xuất: Hai Bà Trưng
Đang trích xuất: Lý Nam Đế
Đang trích xuất: Triệu Việt Vương
Đang trích xuất: Lý Thiên Bảo
Đang trích xuất: Hậu Lý Nam Đế
Đang trích xuất: Lý Sư Lợi
Đang trích xuất: Mai Hắc Đế
Đang trích xuất: Mai Thúc Huy
Đang trích xuất: Mai Kỳ Sơn
Đang trích xuất: Phùng Hưng
Đang trích xuất: Khúc Thừa Dụ
Đang trích xuất: Khúc Hạo
Đang trích xuất: Khúc Thừa Mỹ
Đang trích xuất: Dương Đình Nghệ
Đang trích xuất: Kiều Công Tiễn
Đang trích xuất: Lý Thái Tổ
Đang trích xuất: Lý Thái Tông
Đang trích xuất: Lý Thánh Tông
Đang trích xuất: Lý Nhân Tông
Đang trích xuất: Lý Thần Tông
Đang trích xuất: Lý Anh Tông
Đang trích xuất: Lý Cao Tông
Đang trích xuất: Lý Thẩm
Đang trích xuất: Lý Huệ Tông
Đang trích xuất: Lý Nguyên Hoàng